# P&L attribution: decomposing daily MTM changes by risk factor

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb` and `02_pricing/pricing_across_asset_classes.ipynb`. This notebook assumes you can construct instruments, `MarketContext`, and use the pricing pipeline.

P&L attribution answers: **"Why did my position's value change from T₀ to T₁?"** by isolating the contribution of each market factor.

## Overview

finstack-quant supports four attribution methodologies:

| Method | How it works | Trade-offs |
|---|---|---|
| **Parallel** | Isolate each factor independently (restore T₀ for one factor, keep T₁ for the rest) | Cross-effects go to residual; factors may not sum exactly |
| **Waterfall** | Apply factors sequentially in a specified order | Guarantees sum = total P&L; order matters |
| **Metrics-based** | Linear approximation using pre-computed sensitivities (DV01, CS01, Vega, Theta…) | Fast; less accurate for large moves |
| **Taylor** | Bump-and-reprice sensitivities × observed market moves; optional second-order | Factor-additive; independent of application order |

All four return a `PnlAttribution` object that decomposes total P&L into:
- **Carry** (theta + accruals)
- **Rates curves** (discount & forward)
- **Credit curves** (hazard/spread)
- **Inflation curves**
- **Correlations** (structured credit)
- **FX**
- **Volatility**
- **Cross-factor** interactions
- **Model parameters** (prepay, default, recovery, conversion)
- **Market scalars** (dividends, equity/commodity prices, inflation indices)
- **Residual** (unexplained)

## Imports

In [1]:
import json
import math
from datetime import date

import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF, instrument_fixtures
from finstack_quant.core.market_data import DiscountCurve, HazardCurve, MarketContext
from finstack_quant.attribution import (
    PnlAttribution,
    attribute_pnl,
    attribute_pnl_from_spec,
    default_attribution_metrics,
    default_waterfall_order,
    validate_attribution_json,
)

print("Imported attribution helpers.")
print("Default waterfall order:", default_waterfall_order())
print("Metrics-based metrics:", default_attribution_metrics()[:5], "...")


Imported attribution helpers.
Default waterfall order: ['Carry', 'RatesCurves', 'CreditCurves', 'InflationCurves', 'Correlations', 'Fx', 'Volatility', 'ModelParameters', 'MarketScalars']
Metrics-based metrics: ['theta', 'dv01', 'cs01', 'bucketed_cs01', 'vega'] ...


## Setup: instrument and two market snapshots

We'll attribute the P&L of a **USD corporate bond** — the shared `fixed_bond` fixture, 4.5% semi-annual, maturing 2029-01-15 — between two consecutive dates. Between T₀ and T₁:

- The discount curve shifts **+5 bp** in parallel (continuously-compounded zero rates) — a rates sell-off
- Hazard rates shift **+10 bp** in parallel — credit widening. At a 40% recovery this is roughly **+6 bp of par CDS spread**, since `spread ≈ hazard × (1 − recovery)`
- One day of carry accrues

This lets us see carry, rates, and credit factors clearly separated.

An attribution result is only as meaningful as the shock that produced it. So we build T₁ from T₀ by an **exact, stated transformation** and then **verify the realised shift** against the curves themselves before running any attribution. Confirming that the market you built is the market you meant to build is a habit worth keeping — a mistyped discount factor shows up as a plausible-looking rates P&L, not as an error.

In [2]:
AS_OF_T0 = DEMO_AS_OF.isoformat()
AS_OF_T1 = "2025-01-16"

# The shared fixed-rate bond fixture: 4.5% semi-annual, issued 2024-01-15,
# maturing 2029-01-15. Reusing it keeps this notebook consistent with the rest
# of the examples instead of re-typing a wire payload.
bond_id, bond_spec = instrument_fixtures.fixed_bond(1)

# The fixture is rates-only. Attach a credit curve so the attribution engine has
# a credit factor to isolate.
bond_spec["spec"]["credit_curve_id"] = "CORP-HAZARD"

bond_json = json.dumps(bond_spec)
print(f"Instrument: {bond_id} — {bond_spec['spec']['cashflow_spec']['Fixed']['rate']} coupon, matures {bond_spec['spec']['maturity']}")
print("Instrument JSON ready:", len(bond_json), "chars")

Instrument: BOND-FIXED-1 — 0.045 coupon, matures 2029-01-15
Instrument JSON ready: 647 chars


**What to expect from carry on this bond.** T₀ (`2025-01-15`) is a **coupon date** — the bond pays semi-annually from its 2024-01-15 issue. Carry therefore captures both the **realised coupon** and the **PV roll-down** over the day, so it will be dominated by roughly half a year's coupon (≈ USD 22.6k on a USD 1mm notional at 4.5%) rather than by a single day's accrual. On a non-coupon date you would instead see carry of a few hundred dollars: one day of accrual plus roll-down.

**What to expect from credit.** Because the bond carries a `credit_curve_id`, moving the hazard curve between T₀ and T₁ produces a **non-zero `credit_curves_pnl`** that is isolated from the rates move. If you drop the credit curve, that term collapses to zero and the same P&L is absorbed by rates.

In [3]:
base_t0 = date.fromisoformat(AS_OF_T0)
base_t1 = date.fromisoformat(AS_OF_T1)

# The shock we intend to apply, stated once and used to build T1.
RATE_SHIFT = 0.0005  # +5 bp on continuously-compounded zero rates
HAZARD_SHIFT = 0.0010  # +10 bp on hazard rates

DISCOUNT_KNOTS = [
    (0.0, 1.0), (0.5, 0.980), (1.0, 0.960), (2.0, 0.920),
    (3.0, 0.880), (5.0, 0.800), (10.0, 0.650),
]
HAZARD_KNOTS = [
    (0.5, 0.010), (1.0, 0.012), (2.0, 0.015),
    (3.0, 0.018), (5.0, 0.022), (10.0, 0.028),
]

# --- T0 market ---
disc_t0 = DiscountCurve("USD-OIS", base_t0, DISCOUNT_KNOTS, day_count="act_365f")
haz_t0 = HazardCurve("CORP-HAZARD", base_t0, HAZARD_KNOTS, day_count="act_365f")

mc_t0 = MarketContext()
mc_t0.insert(disc_t0)
mc_t0.insert(haz_t0)
market_t0_json = mc_t0.to_json()

# --- T1 market: derived from T0, not re-typed ---
# A parallel shift of s on the zero rate z(t) is exactly df(t) -> df(t) * exp(-s * t),
# because z(t) = -ln(df(t)) / t. Deriving the knots this way means the realised
# shock is the shock we declared above, to full precision.
disc_t1 = DiscountCurve(
    "USD-OIS",
    base_t1,
    [(t, df if t == 0.0 else df * math.exp(-RATE_SHIFT * t)) for t, df in DISCOUNT_KNOTS],
    day_count="act_365f",
)
haz_t1 = HazardCurve(
    "CORP-HAZARD",
    base_t1,
    [(t, h + HAZARD_SHIFT) for t, h in HAZARD_KNOTS],
    day_count="act_365f",
)

mc_t1 = MarketContext()
mc_t1.insert(disc_t1)
mc_t1.insert(haz_t1)
market_t1_json = mc_t1.to_json()

print(f"T0 market: {len(market_t0_json)} chars")
print(f"T1 market: {len(market_t1_json)} chars")

T0 market: 1749 chars
T1 market: 1850 chars


### Verify the shock before attributing

Read the shift back off the two curves using their own accessors — `DiscountCurve.zero(t)` and `HazardCurve.hazard_rate(t)` — rather than trusting the knots we typed. We check tenors *between* the knots as well as on them, since interpolation is where a shock most often stops being parallel.

Here the two snapshots are two independent observations of the market, so we construct them directly. When you instead want to *derive* a stressed market from a base one — parameterised, composable, and replayable — reach for the scenario DSL (`scenarios.OperationSpec` / `apply_scenario_to_market`), covered in `06_scenarios/`. Whichever route you take, measure the realised shift like this before reading anything into the attribution.

In [4]:
check_tenors = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 7.5, 10.0]

print(f"{'tenor':>6}  {'zero T0':>9}  {'zero T1':>9}  {'shift':>8}   {'hazard shift':>12}")
for t in check_tenors:
    zero_shift_bp = (disc_t1.zero(t) - disc_t0.zero(t)) * 1e4
    hazard_shift_bp = (haz_t1.hazard_rate(t) - haz_t0.hazard_rate(t)) * 1e4
    print(
        f"{t:6.2f}  {disc_t0.zero(t):8.4%}  {disc_t1.zero(t):8.4%}  "
        f"{zero_shift_bp:+7.4f}bp  {hazard_shift_bp:+11.4f}bp"
    )
    # A parallel shock must be the same size at every tenor, not just on average.
    assert abs(zero_shift_bp - RATE_SHIFT * 1e4) < 1e-6, f"rates shock not parallel at {t}y"
    assert abs(hazard_shift_bp - HAZARD_SHIFT * 1e4) < 1e-6, f"credit shock not parallel at {t}y"

print("\nVerified: +5.0000 bp rates and +10.0000 bp hazard, uniform across the curve.")

 tenor    zero T0    zero T1     shift   hazard shift
  0.25   4.0197%   4.0697%  +5.0000bp     +10.0000bp
  0.50   4.0405%   4.0905%  +5.0000bp     +10.0000bp
  0.75   4.0612%   4.1112%  +5.0000bp     +10.0000bp
  1.00   4.0822%   4.1322%  +5.0000bp     +10.0000bp
  1.50   4.1249%   4.1749%  +5.0000bp     +10.0000bp
  2.00   4.1691%   4.2191%  +5.0000bp     +10.0000bp
  3.00   4.2611%   4.3111%  +5.0000bp     +10.0000bp
  4.00   4.3848%   4.4348%  +5.0000bp     +10.0000bp
  5.00   4.4629%   4.5129%  +5.0000bp     +10.0000bp
  7.50   4.4215%   4.4715%  +5.0000bp     +10.0000bp
 10.00   4.3078%   4.3578%  +5.0000bp     +10.0000bp

Verified: +5.0000 bp rates and +10.0000 bp hazard, uniform across the curve.


## Running attribution

`attribute_pnl()` is the main entry point. Pass the instrument, two market snapshots, dates, and a method descriptor — it returns canonical JSON, which you can wrap as `PnlAttribution` when you want the richer Python API:

```python
attr_json = attribute_pnl(instrument_json, market_t0_json, market_t1_json,
                          as_of_t0, as_of_t1, method)
attr = PnlAttribution.from_json(attr_json)
```

The `method` argument selects the methodology: `"Parallel"`, `{"Waterfall": [...]}`, `"MetricsBased"`, or `{"Taylor": {...}}`.

For power-user workflows that need the full JSON envelope round-trip, use `attribute_pnl_from_spec()` and `validate_attribution_json()` instead.

In [5]:
def run(method):
    attr_json = attribute_pnl(bond_json, market_t0_json, market_t1_json, AS_OF_T0, AS_OF_T1, method)
    return PnlAttribution.from_json(attr_json)

print("Ready — run('Parallel'), run({'Waterfall': [...]}) etc.")

Ready — run('Parallel'), run({'Waterfall': [...]}) etc.


## 1. Parallel attribution

Each factor is isolated independently: restore T₀ for one factor while keeping T₁ for all others. Cross-effects appear in the residual.

In [6]:
parallel = run("Parallel")

print(parallel.explain())
print()
print(f"Residual within tolerance: {parallel.residual_within_tolerance()}")
print(f"Method: {parallel.method}")
print(f"Repricings: {parallel.num_repricings}")

Total P&L: USD 18797.15
  ├─ Carry: USD 22621.54 (120.3%)
  │   ├─ Coupon Income: USD 22625.00
  │   ├─ Pull-to-Par: USD 27.79
  │   ├─ Roll-Down: USD -31.25
  ├─ Rates Curves: USD -1740.82 (-9.3%)
  ├─ Credit Curves: USD -2078.24 (-11.1%)
  ├─ Cross-Factor: USD -5.33 (-0.0%)
  │   ├─ Rates×Credit: USD -5.33
  └─ Residual: USD 0.00 (0.0%)

Residual within tolerance: True
Method: Parallel
Repricings: 8


In [7]:
parallel.to_dataframe()

,carry,correlations_pnl,credit_curves_pnl,cross_factor_pnl,currency,fx_pnl,fx_translation_pnl,inflation_curves_pnl,instrument_id,mark_to_market_pnl,...,model_params_pnl,num_repricings,rates_curves_pnl,residual,residual_pct,result_invalid,t0,t1,total_pnl,vol_pnl
0,22621.542172,0.0,-2078.236848,-5.329509,USD,0.0,0.0,0.0,BOND-FIXED-1,-3702.848465,...,0.0,8,-1740.824279,0.0,0.0,False,2025-01-15,2025-01-16,18797.151535,0.0


## 2. Waterfall attribution

Factors are applied sequentially. By construction, the sum of all factor P&Ls equals the total — the residual should be near zero.

The **order matters**: earlier factors absorb more cross-effects. The `default_waterfall_order()` provides a standard starting point.

In [8]:
order = default_waterfall_order()
print("Default waterfall order:", order)

Default waterfall order: ['Carry', 'RatesCurves', 'CreditCurves', 'InflationCurves', 'Correlations', 'Fx', 'Volatility', 'ModelParameters', 'MarketScalars']


In [9]:
waterfall_method = {"Waterfall": ["Carry", "RatesCurves", "CreditCurves", "Fx", "Volatility"]}

waterfall = run(waterfall_method)

print(waterfall.explain())
print()
print(f"Residual: {waterfall.residual:.4f} ({waterfall.residual_pct:.4f}%)")
print(f"Residual within tolerance: {waterfall.residual_within_tolerance()}")

Total P&L: USD 18797.15
  ├─ Carry: USD 22621.54 (120.3%)
  │   ├─ Coupon Income: USD 22625.00
  │   ├─ Pull-to-Par: USD 27.79
  │   ├─ Roll-Down: USD -31.25
  ├─ Rates Curves: USD -1746.15 (-9.3%)
  ├─ Credit Curves: USD -2078.24 (-11.1%)
  └─ Residual: USD 0.00 (0.0%)

Residual: 0.0000 (0.0000%)
Residual within tolerance: True


### Comparing different orderings

Waterfall attribution is order-dependent. Let's see how swapping rates and credit changes the decomposition:

In [10]:
import pandas as pd

order_rates_first = {"Waterfall": ["Carry", "RatesCurves", "CreditCurves"]}
order_credit_first = {"Waterfall": ["Carry", "CreditCurves", "RatesCurves"]}

wf_rates = run(order_rates_first)
wf_credit = run(order_credit_first)

comparison = pd.DataFrame(
    {
        "Rates first": {
            "carry": wf_rates.carry,
            "rates_curves": wf_rates.rates_curves_pnl,
            "credit_curves": wf_rates.credit_curves_pnl,
            "residual": wf_rates.residual,
            "total": wf_rates.total_pnl,
        },
        "Credit first": {
            "carry": wf_credit.carry,
            "rates_curves": wf_credit.rates_curves_pnl,
            "credit_curves": wf_credit.credit_curves_pnl,
            "residual": wf_credit.residual,
            "total": wf_credit.total_pnl,
        },
    }
)
comparison

,Rates first,Credit first
carry,22621.542172,22621.542172
rates_curves,-1746.153788,-1740.824279
credit_curves,-2078.236848,-2083.566357
residual,0.000000,0.000000
total,18797.151535,18797.151535


## 3. Metrics-based attribution

Uses pre-computed first- and second-order sensitivities (Theta, DV01, CS01, Gamma, Convexity, etc.) to approximate the P&L decomposition. This is the fastest method but least accurate for large market moves.

In [11]:
metrics = run("MetricsBased")

print(metrics.explain())
print()
print(f"Residual: {metrics.residual:.4f} ({metrics.residual_pct:.4f}%)")
print(f"Repricings: {metrics.num_repricings}")

Total P&L: USD 18797.15
  ├─ Carry: USD 28327.72 (150.7%)
  │   ├─ Roll-Down: USD 28327.72
  ├─ Rates Curves: USD -1767.73 (-9.4%)
  ├─ Credit Curves: USD -1730.17 (-9.2%)
  ├─ Cross-Factor: USD 5.46 (0.0%)
  │   ├─ Rates×Credit: USD 5.46
  └─ Residual: USD -6038.14 (-32.1%)

Residual: -6038.1356 (-32.1226%)
Repricings: 1


## 4. Taylor attribution

Computes first-order sensitivities via bump-and-reprice at T₀, then multiplies by observed market moves. Optionally includes second-order (gamma/convexity) terms.

Unlike waterfall, the decomposition is factor-additive and does not depend on application order.

In [12]:
taylor_config = {"Taylor": {"include_gamma": False, "rate_bump_bp": 1.0, "credit_bump_bp": 1.0, "vol_bump": 0.01}}

taylor = run(taylor_config)

print(taylor.explain())
print()
print(f"Residual: {taylor.residual:.4f} ({taylor.residual_pct:.4f}%)")
print(f"Repricings: {taylor.num_repricings}")

Total P&L: USD 18797.15
  ├─ Carry: USD 22621.54 (120.3%)
  │   ├─ Coupon Income: USD 22500.00
  │   ├─ Roll-Down: USD 121.54
  ├─ Rates Curves: USD -1769.58 (-9.4%)
  ├─ Credit Curves: USD -1730.17 (-9.2%)
  └─ Residual: USD -324.64 (-1.7%)

Residual: -324.6395 (-1.7271%)
Repricings: 28


In [13]:
taylor_gamma_config = {"Taylor": {"include_gamma": True, "rate_bump_bp": 1.0, "credit_bump_bp": 1.0, "vol_bump": 0.01}}

taylor_gamma = run(taylor_gamma_config)

print("Taylor with gamma (second-order):")
print(taylor_gamma.explain())
print()
print(f"Residual without gamma: {taylor.residual:.4f} ({taylor.residual_pct:.2f}%)")
print(f"Residual with gamma:    {taylor_gamma.residual:.4f} ({taylor_gamma.residual_pct:.2f}%)")

Taylor with gamma (second-order):
Total P&L: USD 18797.15
  ├─ Carry: USD 22621.54 (120.3%)
  │   ├─ Coupon Income: USD 22500.00
  │   ├─ Roll-Down: USD 121.54
  ├─ Rates Curves: USD -1768.48 (-9.4%)
  ├─ Credit Curves: USD -1730.17 (-9.2%)
  └─ Residual: USD -325.75 (-1.7%)

Residual without gamma: -324.6395 (-1.73%)
Residual with gamma:    -325.7459 (-1.73%)


## Method comparison

Stack all four results side-by-side to compare factor decompositions and residual quality.

In [14]:
import pandas as pd

results = {"Parallel": parallel, "Waterfall": waterfall, "MetricsBased": metrics, "Taylor": taylor}

rows = []
for name, attr in results.items():
    rows.append(
        {
            "Method": name,
            "Total P&L": attr.total_pnl,
            "Carry": attr.carry,
            "Rates": attr.rates_curves_pnl,
            "Credit": attr.credit_curves_pnl,
            "FX": attr.fx_pnl,
            "Vol": attr.vol_pnl,
            "Cross": attr.cross_factor_pnl,
            "Residual": attr.residual,
            "Residual %": f"{attr.residual_pct:.2f}%",
            "Repricings": attr.num_repricings,
        }
    )

df = pd.DataFrame(rows).set_index("Method")
df

,Total P&L,Carry,Rates,Credit,FX,Vol,Cross,Residual,Residual %,Repricings
Method,,,,,,,,,,
Parallel,18797.151535,22621.542172,-1740.824279,-2078.236848,0.0,0.0,-5.329509,0.000000,0.00%,8
Waterfall,18797.151535,22621.542172,-1746.153788,-2078.236848,0.0,0.0,0.000000,0.000000,0.00%,9
MetricsBased,18797.151535,28327.720492,-1767.726686,-1730.169145,0.0,0.0,5.462468,-6038.135593,-32.12%,1
Taylor,18797.151535,22621.542172,-1769.581992,-1730.169145,0.0,0.0,0.000000,-324.639499,-1.73%,28


## JSON round-trip and validation

Attribution specs and results serialize to JSON for logging, replay, and cross-language use.

In [15]:
attr_json = parallel.to_json()
print("PnlAttribution JSON (first 500 chars):")
print(attr_json[:500], "...")
print()

roundtrip = PnlAttribution.from_json(attr_json)
print(f"Round-trip total_pnl match: {roundtrip.total_pnl == parallel.total_pnl}")
print(repr(roundtrip))

PnlAttribution JSON (first 500 chars):
{"total_pnl":{"amount":"18797.151535380","currency":"USD"},"mark_to_market_pnl":{"amount":"-3702.848464620","currency":"USD"},"carry":{"amount":"22621.542171829","currency":"USD"},"rates_curves_pnl":{"amount":"-1740.824279106","currency":"USD"},"credit_curves_pnl":{"amount":"-2078.236847996","currency":"USD"},"inflation_curves_pnl":{"amount":"0","currency":"USD"},"correlations_pnl":{"amount":"0","currency":"USD"},"fx_pnl":{"amount":"0","currency":"USD"},"fx_translation_pnl":{"amount":"0","curren ...

Round-trip total_pnl match: True
PnlAttribution(id="BOND-FIXED-1", method=Parallel, total_pnl=18797.15 USD, residual_pct=0.00%)


In [16]:
envelope = json.dumps({
    "schema": "finstack_quant.attribution/1",
    "attribution": {
        "instrument": json.loads(bond_json),
        "market_t0": json.loads(market_t0_json),
        "market_t1": json.loads(market_t1_json),
        "as_of_t0": AS_OF_T0,
        "as_of_t1": AS_OF_T1,
        "method": "Parallel",
    },
})

canonical = validate_attribution_json(envelope)
print(f"Validated spec: {len(canonical)} chars")
print(canonical[:300], "...")

result_json = attribute_pnl_from_spec(envelope)
print(f"\nResult envelope: {len(result_json)} chars")

Validated spec: 3297 chars
{"schema":"finstack_quant.attribution/1","attribution":{"instrument":{"type":"bond","spec":{"id":"BOND-FIXED-1","notional":{"amount":"1000000","currency":"USD"},"issue_date":"2024-01-15","maturity":"2029-01-15","cashflow_spec":{"Fixed":{"coupon_type":"Cash","rate":"0.045","freq":{"count":6,"unit":"m ...

Result envelope: 2047 chars


## Diagnostics and verbose output

`explain_verbose()` shows all factors including those with zero P&L, useful for debugging. Diagnostic `notes` capture warnings from the attribution engine.

In [17]:
print(parallel.explain_verbose())
print()
print("Notes:", parallel.notes)

Total P&L: USD 18797.15
  ├─ Carry: USD 22621.54 (120.3%)
  │   ├─ Coupon Income: USD 22625.00
  │   ├─ Pull-to-Par: USD 27.79
  │   ├─ Roll-Down: USD -31.25
  ├─ Rates Curves: USD -1740.82 (-9.3%)
  ├─ Credit Curves: USD -2078.24 (-11.1%)
  ├─ Inflation Curves: USD 0.00 (0.0%)
  ├─ Correlations: USD 0.00 (0.0%)
  ├─ FX: USD 0.00 (0.0%)
  ├─ Vol: USD 0.00 (0.0%)
  ├─ Cross-Factor: USD -5.33 (-0.0%)
  │   ├─ Rates×Credit: USD -5.33
  ├─ Model Params: USD 0.00 (0.0%)
  ├─ Market Scalars: USD 0.00 (0.0%)
  └─ Residual: USD 0.00 (0.0%)

Notes: []


## Takeaways

- **`attribute_pnl(instrument, mkt_t0, mkt_t1, t0, t1, method)`** is the main entry point — pass instruments and market data directly, get canonical JSON back.
- Wrap that JSON with **`PnlAttribution.from_json(...)`** when you want property accessors, `explain()`, `to_dataframe()`, and JSON round-trip helpers.
- **`attribute_pnl_from_spec(envelope_json)`** is the power-user variant for full JSON round-trip workflows.
- **Parallel** is simplest and order-independent; **Waterfall** guarantees the sum constraint; **Metrics-based** is fastest; **Taylor** gives sensitivity decomposition.
- For large portfolios, use `PnlAttribution.to_dataframe()` and `pd.concat` to build a full book-level attribution table.
- All results serialize to JSON for logging and replay.